# SECCIÓN 3. PREPARACIÓN DE DATOS

Este cuaderno desarrolla los puntos 3 y 4 del trabajo mediante la metodología CRISP-DM. El objetivo es predecir `trip_duration` (segundos) con información disponible al inicio del viaje, evitando fuga de información.


## 3.1 Selección de variables desde el conocimiento del negocio

Se excluyen identificadores, datos personales y variables sin relación operativa (`RecordID`, `FullName`, `Phone`, `ZodiacSign`, `FavoriteColor`, `Hobby`). También se elimina `total_fare`, porque se conoce durante o después del viaje y produciría fuga de información. `VendorID` se descarta por ser casi constante. Se conserva `trip_distance_miles` y se elimina `trip_distance`: ambas representan la misma magnitud, pero ocho valores erróneos de la segunda columna (−5, 500 y 800 km) destruyen su correlación, mientras que el 99,5 % de las filas comparables conserva correctamente la conversión.


In [1]:
import os, warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

# Funciona directamente en Colab y también junto al notebook descargado.
file_path = '/content/Taxi_Trip_Duration.csv'
if not os.path.exists(file_path):
    file_path = 'Taxi_Trip_Duration.csv'

df = pd.read_csv(file_path)
print(f'Dimensiones iniciales: {df.shape[0]} filas y {df.shape[1]} columnas')
display(df.head())


Dimensiones iniciales: 1800 filas y 21 columnas


,RecordID,FullName,Phone,ZodiacSign,FavoriteColor,Hobby,pickup_datetime,pickup_latitude,pickup_longitude,dropoff_latitude,dropoff_longitude,passenger_count,trip_distance,trip_distance_miles,hour,day_of_week,is_weekend,weather,VendorID,trip_duration,total_fare
0,REC00001,Sofia Miller,3.449545e+09,Cancer,Yellow,Cooking,01/06/2023 00:00:00,40.660618,-73.979191,40.669406,-74.003883,2.0,NaN,9.091601,21,4,0,Fog,A,2296.617526,40.229624
1,REC00002,John Gonzalez,3.648211e+09,Capricorn,White,Gaming,01/06/2023 00:17:00,40.717049,-73.911793,40.724527,-73.918776,2.0,11.823875,7.347013,22,4,0,Rain,A,1861.815211,32.842164
2,REC00003,Thomas Perez,3.374371e+09,Aries,Green,Reading,01/06/2023 00:34:00,NaN,-74.006844,40.716247,-74.008732,1.0,5.286519,3.284890,9,1,0,Clear,A,838.891983,15.233691
3,REC00004,Joseph Thomas,3.697255e+09,Libra,Yellow,Hiking,01/06/2023 00:51:00,NaN,-73.971136,40.751094,-73.951873,1.0,5.120515,3.181740,18,3,0,Snow,A,1223.455139,19.487708
4,REC00005,Charles Lopez,3.432576e+09,Sagittarius,Red,Reading,01/06/2023 01:08:00,40.817390,-74.055283,40.835754,-73.997030,NaN,2.633424,1.636333,18,1,0,Clear,A,646.133860,11.904101


In [2]:
# Auditoría de redundancia y calidad de la distancia
valid_dist = df['trip_distance'].notna() & df['trip_distance_miles'].notna()
expected_km = df['trip_distance_miles'] * 1.609344
bad_dist = valid_dist & (df['trip_distance'].sub(expected_km).abs() > 0.02)
conversion_ok = 100 * (valid_dist.sum() - bad_dist.sum()) / valid_dist.sum()

print(f'Correlación original km-millas: {df.trip_distance.corr(df.trip_distance_miles):.4f}')
print(f'Registros inconsistentes: {bad_dist.sum()}')
print(f'Conversión correcta entre filas comparables: {conversion_ok:.2f}%')
display(df.loc[bad_dist, ['trip_distance', 'trip_distance_miles', 'trip_duration']])

exclude = ['RecordID','FullName','Phone','ZodiacSign','FavoriteColor','Hobby',
           'total_fare','VendorID','trip_distance']
df_clean = df.drop(columns=exclude, errors='ignore').copy()
print('Variables inicialmente conservadas:', df_clean.columns.tolist())


Correlación original km-millas: 0.1102
Registros inconsistentes: 8
Conversión correcta entre filas comparables: 99.51%


,trip_distance,trip_distance_miles,trip_duration
739,500.0,1.683960,754.632193
948,500.0,5.091247,1540.159587
963,800.0,5.352428,1204.926759
1024,-5.0,1.908915,522.371788
1189,-5.0,2.225154,487.526546
1203,800.0,2.238030,664.217448
1650,500.0,3.027419,728.600851
1660,-5.0,3.318510,1141.237628


Variables inicialmente conservadas: ['pickup_datetime', 'pickup_latitude', 'pickup_longitude', 'dropoff_latitude', 'dropoff_longitude', 'passenger_count', 'trip_distance_miles', 'hour', 'day_of_week', 'is_weekend', 'weather', 'trip_duration']


## 3.2 Descripción estadística

Se revisan tipos, medidas descriptivas, valores faltantes, duplicados y distribución de la variable objetivo. Esto permite distinguir errores de captura de observaciones extremas plausibles.


In [3]:
print('Tipos de datos:')
display(df.dtypes.to_frame('tipo'))
print('Descripción numérica:')
display(df.describe().T)
print('Descripción categórica:')
display(df.describe(include=['object']).T)

missing = pd.DataFrame({'nulos': df.isna().sum(), 'porcentaje': df.isna().mean()*100})
display(missing[missing.nulos > 0].sort_values('porcentaje', ascending=False))
print('Filas duplicadas:', df.duplicated().sum())
print(f'Asimetría inicial de trip_duration: {df.trip_duration.skew():.4f}')

fig, ax = plt.subplots(1, 2, figsize=(12,4))
ax[0].hist(df.trip_duration, bins=50); ax[0].set_title('trip_duration original'); ax[0].set_xlabel('segundos')
ax[1].hist(np.log1p(df.trip_duration), bins=50, color='orange'); ax[1].set_title('log(1 + trip_duration)')
plt.tight_layout(); plt.show()


Tipos de datos:


,tipo
RecordID,object
FullName,object
Phone,float64
ZodiacSign,object
FavoriteColor,object
Hobby,object
pickup_datetime,object
pickup_latitude,float64
pickup_longitude,float64
dropoff_latitude,float64


Descripción numérica:


,count,mean,std,min,25%,50%,75%,max
Phone,1800.0,3.502309e+09,2.896287e+08,3.000030e+09,3.257613e+09,3.497885e+09,3.747475e+09,3.999212e+09
pickup_latitude,1620.0,4.074959e+01,5.007573e-02,4.055915e+01,4.071596e+01,4.074955e+01,4.078272e+01,4.093801e+01
pickup_longitude,1800.0,-7.397858e+01,4.986388e-02,-7.414992e+01,-7.401224e+01,-7.397975e+01,-7.394396e+01,-7.383761e+01
dropoff_latitude,1800.0,4.074947e+01,5.918821e-02,4.052061e+01,4.071040e+01,4.074950e+01,4.078771e+01,4.095076e+01
dropoff_longitude,1800.0,-7.397925e+01,5.941080e-02,-7.417157e+01,-7.402002e+01,-7.397981e+01,-7.393925e+01,-7.378579e+01
passenger_count,1628.0,2.030713e+00,1.609832e+00,0.000000e+00,1.000000e+00,1.000000e+00,3.000000e+00,2.000000e+01
trip_distance,1628.0,6.944888e+00,3.518067e+01,-5.000000e+00,2.471161e+00,4.269459e+00,6.982435e+00,8.000000e+02
trip_distance_miles,1800.0,3.144034e+00,2.219837e+00,2.633679e-02,1.503807e+00,2.631996e+00,4.321675e+00,1.483912e+01
hour,1800.0,1.155222e+01,6.931456e+00,0.000000e+00,6.000000e+00,1.200000e+01,1.800000e+01,2.300000e+01
day_of_week,1800.0,3.022778e+00,2.006945e+00,0.000000e+00,1.000000e+00,3.000000e+00,5.000000e+00,6.000000e+00


Descripción categórica:


,count,unique,top,freq
RecordID,1800,1800,REC00001,1
FullName,1800,393,Robert Garcia,11
ZodiacSign,1800,12,Taurus,179
FavoriteColor,1800,10,Gray,197
Hobby,1800,10,Gardening,194
pickup_datetime,1800,1800,01/06/2023 00:00:00,1
weather,1620,4,Clear,955
VendorID,1800,3,A,1749


,nulos,porcentaje
pickup_latitude,180,10.000000
weather,180,10.000000
passenger_count,172,9.555556
trip_distance,172,9.555556


Filas duplicadas: 0
Asimetría inicial de trip_duration: 20.7623


## 3.3 Limpieza de atípicos

Se aplican reglas de negocio transparentes: duración entre 60 segundos y 4 horas, distancia positiva y no mayor de 100 millas, y entre 1 y 6 pasajeros enteros. Los nulos de pasajeros se conservan temporalmente para imputarlos dentro del pipeline. Esta decisión elimina errores evidentes sin aplicar recortes estadísticos arbitrarios.


In [4]:
before = len(df_clean)
rule_duration = df_clean.trip_duration.between(60, 14400)
rule_distance = df_clean.trip_distance_miles.gt(0) & df_clean.trip_distance_miles.le(100)
rule_passenger = (df_clean.passenger_count.isna() |
                  (df_clean.passenger_count.between(1,6) & df_clean.passenger_count.mod(1).eq(0)))

audit = pd.DataFrame({
    'regla': ['Duración fuera de 60–14.400 s', 'Distancia fuera de 0–100 millas', 'Pasajeros fuera de 1–6 enteros'],
    'registros_detectados': [(~rule_duration).sum(), (~rule_distance).sum(), (~rule_passenger).sum()]
})
display(audit)
df_clean = df_clean[rule_duration & rule_distance & rule_passenger].copy()
print(f'Antes: {before}; después: {len(df_clean)}; eliminados: {before-len(df_clean)} ({100*(before-len(df_clean))/before:.2f}%)')


,regla,registros_detectados
0,Duración fuera de 60–14.400 s,8
1,Distancia fuera de 0–100 millas,0
2,Pasajeros fuera de 1–6 enteros,8


Antes: 1800; después: 1784; eliminados: 16 (0.89%)


## 3.4 Limpieza de nulos

La partición 70/30 se realiza antes de aprender cualquier imputación. En cada pliegue de validación cruzada, las variables numéricas se imputan con la mediana y `weather` con la categoría `Desconocido`. Así, los estadísticos del conjunto de prueba nunca participan en el entrenamiento.

## 3.5 Creación de nuevas variables y One-Hot Encoding

`pickup_datetime` se convierte desde el formato día/mes/año. A partir de ella se crean `pickup_hour`, `pickup_day_of_week` y `pickup_is_weekend`. Se sustituyen las columnas temporales originales porque no coinciden con la fecha (la hora solo coincide en cerca del 4,6 % de los registros). No se crea el mes porque todos los registros pertenecen a junio y sería una variable constante. `weather` se transforma mediante One-Hot Encoding dentro del pipeline.


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

dt = pd.to_datetime(df_clean['pickup_datetime'], dayfirst=True, errors='coerce')
print('Fechas inválidas:', dt.isna().sum())
print(f'Coincidencia de hour original con fecha: {(df_clean.hour == dt.dt.hour).mean()*100:.2f}%')
print(f'Coincidencia de day_of_week original con fecha: {(df_clean.day_of_week == dt.dt.dayofweek).mean()*100:.2f}%')

df_clean['pickup_hour'] = dt.dt.hour
df_clean['pickup_day_of_week'] = dt.dt.dayofweek
df_clean['pickup_is_weekend'] = dt.dt.dayofweek.isin([5,6]).astype(int)
df_clean = df_clean.drop(columns=['pickup_datetime','hour','day_of_week','is_weekend'])

X = df_clean.drop(columns='trip_duration')
y = df_clean['trip_duration']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.30, random_state=42)

num_cols = X_train.select_dtypes(include=np.number).columns.tolist()
cat_cols = X_train.select_dtypes(exclude=np.number).columns.tolist()
num_pipe = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
cat_pipe = Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value='Desconocido')),
                     ('onehot', OneHotEncoder(handle_unknown='ignore'))])
preprocessor = ColumnTransformer([('num', num_pipe, num_cols), ('cat', cat_pipe, cat_cols)])

print('Entrenamiento:', X_train.shape, 'Prueba:', X_test.shape)
print('Numéricas:', num_cols); print('Categóricas:', cat_cols)


Fechas inválidas: 0
Coincidencia de hour original con fecha: 4.60%
Coincidencia de day_of_week original con fecha: 14.41%
Entrenamiento: (1248, 10) Prueba: (536, 10)
Numéricas: ['pickup_latitude', 'pickup_longitude', 'dropoff_latitude', 'dropoff_longitude', 'passenger_count', 'trip_distance_miles', 'pickup_hour', 'pickup_day_of_week', 'pickup_is_weekend']
Categóricas: ['weather']


## 3.6 Análisis de relaciones

El análisis lineal utiliza Pearson. Para detectar asociaciones monótonas se utiliza Spearman y para relaciones no lineales se calcula información mutua sobre los datos de entrenamiento ya imputados y codificados. `trip_distance_miles` es la señal predictiva principal; las demás variables aportan patrones temporales, espaciales y climáticos complementarios.


In [6]:
from sklearn.feature_selection import mutual_info_regression

explore = X_train.select_dtypes(include=np.number).copy()
explore['trip_duration'] = y_train
pearson = explore.corr(method='pearson').trip_duration.drop('trip_duration').sort_values(key=abs, ascending=False)
spearman = explore.corr(method='spearman').trip_duration.drop('trip_duration').sort_values(key=abs, ascending=False)
display(pd.DataFrame({'Pearson': pearson, 'Spearman': spearman}))

X_mi = preprocessor.fit_transform(X_train)
if hasattr(X_mi, 'toarray'): X_mi = X_mi.toarray()
feature_names = preprocessor.get_feature_names_out()
mi = pd.Series(mutual_info_regression(X_mi, y_train, random_state=42), index=feature_names).sort_values(ascending=False)
print('Información mutua (relaciones no lineales):')
display(mi.head(12).to_frame('información_mutua'))

plt.figure(figsize=(7,4)); plt.scatter(X_train.trip_distance_miles, y_train, alpha=.35)
plt.xlabel('Distancia (millas)'); plt.ylabel('Duración (s)'); plt.title('Relación distancia-duración'); plt.show()


,Pearson,Spearman
dropoff_latitude,0.007962,-0.013471
dropoff_longitude,0.009093,-0.014495
passenger_count,0.001809,-0.015445
pickup_day_of_week,-0.026685,-0.041484
pickup_hour,0.015554,0.012960
pickup_is_weekend,-0.033626,-0.042715
pickup_latitude,0.003404,-0.005771
pickup_longitude,0.018440,-0.004924
trip_distance_miles,0.891182,0.852557


Información mutua (relaciones no lineales):


,información_mutua
num__trip_distance_miles,0.751193
num__dropoff_latitude,0.034680
cat__weather_Clear,0.031046
cat__weather_Snow,0.030323
num__pickup_is_weekend,0.019119
cat__weather_Fog,0.016269
cat__weather_Rain,0.004737
num__pickup_latitude,0.003928
num__passenger_count,0.001197
num__pickup_longitude,0.000000


## 3.7 Reducción de dimensiones

PCA se evalúa únicamente sobre las variables numéricas de entrenamiento escaladas. Se reporta cuántos componentes hacen falta para explicar al menos el 90 % de la varianza. No se incorpora al modelo final porque el número de predictores es pequeño, no existe multicolinealidad severa después de eliminar la distancia redundante y conservar las variables facilita la interpretación.


In [7]:
from sklearn.decomposition import PCA

X_num = num_pipe.fit_transform(X_train[num_cols])
pca = PCA().fit(X_num)
cumvar = np.cumsum(pca.explained_variance_ratio_)
n90 = np.argmax(cumvar >= .90) + 1
print(f'Primer componente: {pca.explained_variance_ratio_[0]*100:.2f}%')
print(f'Componentes para explicar al menos 90%: {n90} de {len(num_cols)}')
plt.plot(range(1,len(cumvar)+1), cumvar, marker='o'); plt.axhline(.90,color='red',ls='--')
plt.xlabel('Componentes'); plt.ylabel('Varianza acumulada'); plt.title('Evaluación de PCA'); plt.show()


Primer componente: 21.48%
Componentes para explicar al menos 90%: 6 de 9


## 3.8 Balanceo

No aplica. SMOTE, sobremuestreo y submuestreo están diseñados para clases categóricas; `trip_duration` es una variable continua y el ejercicio corresponde a regresión.

# SECCIÓN 4. MODELAMIENTO Y EVALUACIÓN

## 4.1 Configuración y selección de métodos de Machine Learning

### 4.1.1 Cinco modelos clásicos

Regresión lineal (línea base interpretable), KNN (vecindades), árbol de decisión (reglas no lineales), SVR (margen y kernel) y MLP (red neuronal).

### 4.1.2 Tres ensambles

VotingRegressor (votación/promedio), BaggingRegressor (bagging) e HistGradientBoostingRegressor (boosting). Todos se integran con el mismo pipeline de preprocesamiento.


In [8]:
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import VotingRegressor, BaggingRegressor, HistGradientBoostingRegressor

classic_models = {
 'Regresión lineal': LinearRegression(),
 'KNN': KNeighborsRegressor(),
 'Árbol de decisión': DecisionTreeRegressor(random_state=42),
 'SVR': SVR(),
 'MLP': MLPRegressor(random_state=42, max_iter=1000, early_stopping=True)
}
ensemble_models = {
 'Voting': VotingRegressor([('lr',LinearRegression()),
                            ('tree',DecisionTreeRegressor(max_depth=5,random_state=42)),
                            ('hgb',HistGradientBoostingRegressor(random_state=42))]),
 'Bagging': BaggingRegressor(estimator=DecisionTreeRegressor(random_state=42), random_state=42),
 'Boosting': HistGradientBoostingRegressor(random_state=42)
}
models = {**classic_models, **ensemble_models}
display(pd.DataFrame({'grupo': ['Clásico']*5+['Ensamble']*3,
                      'modelo': list(models.keys())}))


,grupo,modelo
0,Clásico,Regresión lineal
1,Clásico,KNN
2,Clásico,Árbol de decisión
3,Clásico,SVR
4,Clásico,MLP
5,Ensamble,Voting
6,Ensamble,Bagging
7,Ensamble,Boosting


## 4.2 Ajuste de hiperparámetros con el 70 % de los datos haciendo cross validation

### 4.2.1 Justificación de la métrica

MAE es la métrica principal porque expresa el error directamente en segundos y es menos sensible a errores grandes aislados que RMSE. También se reportan RMSE, que penaliza fuertemente errores grandes, y R², que muestra la proporción de variabilidad explicada. MAPE no se usa porque las duraciones pequeñas pueden inflar artificialmente el porcentaje.

### 4.2.2 Ajuste de modelos clásicos

Se utiliza GridSearchCV con cinco pliegues únicamente sobre el 70 % de entrenamiento. Se ajustan KNN, árbol, SVR y MLP; la regresión lineal sirve como línea base sin hiperparámetros relevantes.

### 4.2.3 Ajuste de dos modelos de ensamble

Se ajustan Bagging y Boosting, tal como pide el enunciado. Voting se evalúa con una combinación previamente definida para representar el ensamble de votación.


In [9]:
from sklearn.model_selection import GridSearchCV, KFold, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

param_grids = {
 'KNN': {'model__n_neighbors':[3,5,7,10,15], 'model__weights':['uniform','distance'], 'model__p':[1,2]},
 'Árbol de decisión': {'model__max_depth':[5,10,15,None], 'model__min_samples_split':[2,5,10], 'model__min_samples_leaf':[1,2,4]},
 'SVR': {'model__C':[1,10,50,100], 'model__epsilon':[.05,.1,.2], 'model__kernel':['linear','rbf']},
 'MLP': {'model__hidden_layer_sizes':[(50,),(100,),(50,50)], 'model__alpha':[.0001,.001], 'model__learning_rate_init':[.001,.01]},
 'Bagging': {'model__n_estimators':[20,50,100], 'model__max_samples':[.5,.75,1.0], 'model__max_features':[.7,1.0]},
 'Boosting': {'model__learning_rate':[.03,.05,.1], 'model__max_iter':[100,200], 'model__max_leaf_nodes':[15,31], 'model__l2_regularization':[0,.1]}
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)
best_models, cv_rows = {}, []
for name, model in models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('model', model)])
    start = time.time()
    if name in param_grids:
        search = GridSearchCV(pipe, param_grids[name], scoring='neg_mean_absolute_error', cv=cv, n_jobs=-1)
        search.fit(X_train, y_train)
        best, cv_mae, params = search.best_estimator_, -search.best_score_, search.best_params_
    else:
        cv_mae = -cross_val_score(pipe, X_train, y_train, scoring='neg_mean_absolute_error', cv=cv, n_jobs=-1).mean()
        best, params = pipe.fit(X_train, y_train), 'Configuración definida'
    best_models[name] = best
    cv_rows.append([name, cv_mae, params, time.time()-start])

cv_results = pd.DataFrame(cv_rows, columns=['Modelo','MAE CV (s)','Mejores parámetros','Tiempo (s)']).sort_values('MAE CV (s)')
display(cv_results)


,Modelo,MAE CV (s),Mejores parámetros,Tiempo (s)
0,Regresión lineal,156.640869,Configuración definida,1.131951
3,SVR,157.095246,"{'model__C': 50, 'model__epsilon': 0.05, 'mode...",0.878510
4,MLP,160.463772,"{'model__alpha': 0.001, 'model__hidden_layer_s...",5.903478
5,Voting,163.441908,Configuración definida,0.320149
7,Boosting,167.825092,"{'model__l2_regularization': 0.1, 'model__lear...",2.876320
6,Bagging,168.218928,"{'model__max_features': 1.0, 'model__max_sampl...",2.310259
2,Árbol de decisión,175.649003,"{'model__max_depth': 5, 'model__min_samples_le...",0.434001
1,KNN,232.740220,"{'model__n_neighbors': 7, 'model__p': 2, 'mode...",1.276312


## 4.3 Medida de calidad del modelo

### 4.3.1 Evaluación con el set de pruebas

Los modelos ya seleccionados se evalúan una sola vez sobre el 30 % reservado. Esto proporciona una estimación independiente de generalización.


In [10]:
test_rows = []
predictions = {}
for name, model in best_models.items():
    pred = model.predict(X_test); predictions[name] = pred
    test_rows.append([name, mean_absolute_error(y_test,pred),
                      np.sqrt(mean_squared_error(y_test,pred)), r2_score(y_test,pred)])

test_results = pd.DataFrame(test_rows, columns=['Modelo','MAE (s)','RMSE (s)','R²']).sort_values('MAE (s)')
display(test_results)
plt.figure(figsize=(9,4)); plt.barh(test_results['Modelo'], test_results['MAE (s)'])
plt.gca().invert_yaxis(); plt.xlabel('MAE (segundos)'); plt.title('Comparación en prueba'); plt.tight_layout(); plt.show()


,Modelo,MAE (s),RMSE (s),R²
3,SVR,148.438293,192.538022,0.867110
0,Regresión lineal,149.376529,192.975158,0.866506
4,MLP,151.672215,197.211807,0.860580
6,Bagging,155.612378,200.337551,0.856126
5,Voting,156.283879,201.193876,0.854893
7,Boosting,159.618663,206.451839,0.847210
2,Árbol de decisión,171.582892,218.440047,0.828950
1,KNN,209.460752,271.015622,0.736703


### 4.3.2 Selección del modelo con mejor desempeño

El modelo final se selecciona exclusivamente por el menor MAE en prueba, de acuerdo con la métrica principal definida antes de observar los resultados. La celda siguiente genera automáticamente la conclusión para evitar inconsistencias si el cuaderno se vuelve a ejecutar.


In [11]:
winner = test_results.iloc[0]
print(f"Modelo seleccionado: {winner['Modelo']}")
print(f"MAE: {winner['MAE (s)']:.2f} segundos ({winner['MAE (s)']/60:.2f} minutos)")
print(f"RMSE: {winner['RMSE (s)']:.2f} segundos")
print(f"R²: {winner['R²']:.4f}")

plt.figure(figsize=(6,5))
plt.scatter(y_test, predictions[winner['Modelo']], alpha=.45)
limits=[min(y_test.min(), predictions[winner['Modelo']].min()), max(y_test.max(), predictions[winner['Modelo']].max())]
plt.plot(limits, limits, 'r--'); plt.xlabel('Duración real (s)'); plt.ylabel('Predicción (s)')
plt.title(f"Valores reales vs. predichos — {winner['Modelo']}"); plt.tight_layout(); plt.show()


Modelo seleccionado: SVR
MAE: 148.44 segundos (2.47 minutos)
RMSE: 192.54 segundos
R²: 0.8671


## Conclusión

El flujo completo evita fuga de información, corrige los errores detectados, utiliza variables disponibles al inicio del viaje y compara ocho métodos bajo el mismo esquema de validación. El MAE permite traducir el desempeño final a segundos y minutos. Como limitación, el dataset no incluye tráfico en tiempo real, cierres viales ni incidentes, factores que podrían explicar parte del error residual.

**Enlace de Google Colab:** pegar aquí el enlace compartido con permiso “Cualquier persona que tenga el enlace”.


# SECCIÓN 5. DESPLIEGUE DEL MODELO CON STREAMLIT

Como etapa final, el modelo seleccionado se integró en una aplicación web construida con Streamlit y preparada para Streamlit Community Cloud. La aplicación convierte el análisis académico en una herramienta interactiva que puede ser utilizada sin ejecutar código.

**Repositorio de la aplicación:** https://github.com/StevenCu07/Taxi_Drip

La aplicación incorpora cuatro módulos:

1. **Predicción individual:** permite ingresar distancia en kilómetros o millas, pasajeros, clima, fecha, hora y coordenadas para estimar la duración.
2. **Procesamiento masivo:** acepta nuevos archivos CSV, realiza conversión bidireccional entre kilómetros y millas, genera predicciones y permite descargar los resultados.
3. **Comparación con datos reales:** cuando el archivo contiene `trip_duration`, calcula MAE, RMSE y R² y genera un gráfico real frente a predicho.
4. **Exploración y transparencia:** presenta las distribuciones, las variables, las limitaciones y permite seleccionar cualquiera de los ocho modelos evaluados. SVR aparece como recomendado por ser el ganador según MAE.

El despliegue utiliza los mismos pipelines y configuraciones de la sección 4. Los ocho modelos se entrenan al iniciar la aplicación y Streamlit los conserva en caché. SVR (`C=50`, `epsilon=0.05`, kernel lineal) continúa siendo el modelo oficial y predeterminado, mientras que los demás se ofrecen para comparación interactiva.


In [12]:
# Ejemplo de la lógica utilizada por la aplicación para una predicción
ejemplo_despliegue = X_test.iloc[[0]].copy()
prediccion_despliegue = best_models['SVR'].predict(ejemplo_despliegue)[0]

print('Variables recibidas por la aplicación:')
display(ejemplo_despliegue)
print(f'Duración estimada: {prediccion_despliegue:.2f} segundos')
print(f'Equivalente: {prediccion_despliegue/60:.2f} minutos')


Variables recibidas por la aplicación:


,pickup_latitude,pickup_longitude,dropoff_latitude,dropoff_longitude,passenger_count,trip_distance_miles,weather,pickup_hour,pickup_day_of_week,pickup_is_weekend
1388,40.752611,-74.042665,40.716391,-74.083951,NaN,5.559,Clear,9,5,1


Duración estimada: 1356.08 segundos
Equivalente: 22.60 minutos


## Archivos para el despliegue

- `app.py`: interfaz, entrenamiento, predicción individual, carga masiva y visualizaciones.
- `Taxi_Trip_Duration.csv`: datos utilizados para entrenar el pipeline.
- `requirements.txt`: versiones compatibles de Streamlit, pandas, NumPy, scikit-learn y Plotly.
- `.streamlit/config.toml`: configuración visual y del servidor.
- `Trabajo_final.ipynb`: cuaderno completo del proyecto.

## Procedimiento en Streamlit Community Cloud

1. Ingresar a https://share.streamlit.io/ con la cuenta de GitHub.
2. Seleccionar **Create app**.
3. Elegir el repositorio `StevenCu07/Taxi_Drip` y la rama `main`.
4. Seleccionar `app.py` como archivo principal.
5. Presionar **Deploy** y esperar la instalación de dependencias.

**Enlace público de la aplicación:** pegar aquí la URL generada por Streamlit Community Cloud.

La aplicación tiene un propósito académico. No incluye información de tráfico en tiempo real, incidentes ni cierres viales, por lo que sus resultados deben interpretarse como estimaciones y no como tiempos garantizados.
